In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import re

### Data

In [2]:
DATA_URL = "https://github.com/adamclerget/Project_data_mining_energy/raw/refs/heads/main/manchester_epc_legal.parquet"

print("downloading...")

try:
    df_certificates = pd.read_parquet(DATA_URL, engine="pyarrow")
    
    print("\nDataset has been successfully charged")
    print(f"-> Dimension : {df_certificates.shape[0]} rows and {df_certificates.shape[1]} columns.")

    display(df_certificates.head(3))
    
except Exception as e:
    print(f"\n❌ error")
    print(f"details : {e}")

downloading...

Dataset has been successfully charged
-> Dimension : 334270 rows and 93 columns.


,LMK_KEY,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,BUILDING_REFERENCE_NUMBER,CURRENT_ENERGY_RATING,POTENTIAL_ENERGY_RATING,CURRENT_ENERGY_EFFICIENCY,POTENTIAL_ENERGY_EFFICIENCY,...,CONSTITUENCY_LABEL,POSTTOWN,CONSTRUCTION_AGE_BAND,LODGEMENT_DATETIME,TENURE,FIXED_LIGHTING_OUTLETS_COUNT,LOW_ENERGY_FIXED_LIGHT_COUNT,UPRN,UPRN_SOURCE,REPORT_TYPE
0,20e4f9426b3cec3b7a4ba942710ce71ea7d358f8d25e67...,428 St. Marys Road,None,None,M40 0DE,10003640465,C,B,70,85,...,Manchester Central,MANCHESTER,England and Wales: 1930-1949,2022-11-23 13:53:04,Owner-occupied,8.0,NaN,77038448.0,Energy Assessor,100
1,545383801032010092719131085268408,"20, Parkleigh Drive",None,None,M40 3RY,895420868,D,D,64,68,...,Manchester Central,MANCHESTER,England and Wales: 1930-1949,2010-09-27 19:13:10,owner-occupied,NaN,NaN,77041216.0,Address Matched,100
2,227228561752014092907473698240559,"22, Silverwell Street",None,None,M40 1PA,5991187568,D,B,55,84,...,Manchester Central,MANCHESTER,England and Wales: 1900-1929,2014-09-29 07:47:36,rental (private),6.0,0.0,77035714.0,Address Matched,100


In [3]:
useful_columns = [
    'PROPERTY_TYPE',            # Type of place (house, flat...)
    'BUILT_FORM',               # Configuration
    'CONSTRUCTION_AGE_BAND',
    'TOTAL_FLOOR_AREA',
    'MAINHEAT_DESCRIPTION',     # Type of heating
    'WALLS_DESCRIPTION',
    'ROOF_DESCRIPTION',
    'WINDOWS_DESCRIPTION',
    'CURRENT_ENERGY_RATING',    # (A-G)
    'CURRENT_ENERGY_EFFICIENCY', # Score (1-100), "Based on cost of energy, i.e. energy required for space heating, water heating and lighting [in kWh/year] multiplied by fuel costs. (£/m²/year where cost is derived from kWh).
    'ENERGY_CONSUMPTION_CURRENT' # Current estimated total energy consumption for the property in a 12 month period (kWh/m2).
]

In [4]:
df_model = df_certificates[useful_columns].copy()

efficient_class = ['A', 'B', 'C']
df_model['target_is_efficient'] = df_model['CURRENT_ENERGY_RATING'].isin(efficient_class).astype(int)

In [5]:
df_model

,PROPERTY_TYPE,BUILT_FORM,CONSTRUCTION_AGE_BAND,TOTAL_FLOOR_AREA,MAINHEAT_DESCRIPTION,WALLS_DESCRIPTION,ROOF_DESCRIPTION,WINDOWS_DESCRIPTION,CURRENT_ENERGY_RATING,CURRENT_ENERGY_EFFICIENCY,ENERGY_CONSUMPTION_CURRENT,target_is_efficient
0,House,Mid-Terrace,England and Wales: 1930-1949,81.00,"Boiler and radiators, mains gas","Solid brick, as built, no insulation (assumed)","Pitched, 150 mm loft insulation",Fully double glazed,C,70,209,1
1,House,Semi-Detached,England and Wales: 1930-1949,88.92,"Boiler and radiators, mains gas","Cavity wall, filled cavity","Pitched, 200 mm loft insulation",Fully double glazed,D,64,279,0
2,House,Mid-Terrace,England and Wales: 1900-1929,59.00,"Boiler and radiators, mains gas","Cavity wall, as built, no insulation (assumed)","Pitched, no insulation (assumed)",Fully double glazed,D,55,322,0
3,Flat,Enclosed End-Terrace,England and Wales: 1967-1975,156.00,"Room heaters, electric","Cavity wall, as built, no insulation (assumed)","Pitched, limited insulation (assumed)",Fully double glazed,G,11,505,0
4,House,End-Terrace,England and Wales: 1967-1975,120.60,"Boiler and radiators, mains gas","Cavity wall, filled cavity","Pitched, 250mm loft insulation",Fully double glazed,C,74,182,1
...,...,...,...,...,...,...,...,...,...,...,...,...
334265,Flat,Not Recorded,England and Wales: 1900-1929,76.00,"Boiler and radiators, mains gas","Cavity wall, filled cavity","Pitched, 300 mm loft insulation",Fully double glazed,C,78,120,1
334266,House,End-Terrace,England and Wales: 1976-1982,81.00,"Boiler and radiators, mains gas","Cavity wall, filled cavity","Pitched, 175 mm loft insulation",Fully double glazed,C,71,154,1
334267,House,Mid-Terrace,England and Wales: 1900-1929,75.00,"Boiler and radiators, mains gas","Solid brick, as built, no insulation (assumed)","Pitched, insulated (assumed)",High performance glazing,D,66,225,0
334268,Flat,Not Recorded,England and Wales: 1930-1949,65.00,"Boiler and radiators, mains gas","Cavity wall, filled cavity","Pitched, 300 mm loft insulation",Fully double glazed,C,74,143,1


### Variables Transformation

#### MAINHEAT_DESCRIPTION

In [6]:
# 1. Cleaning names
heat_desc = df_model["MAINHEAT_DESCRIPTION"].str.lower().fillna("")

# 2. Dict of concepts
keywords_mapping_v2 = {
    # ENERGY 
    # adding welsh word "nwy" (gas)
    "heat_energy_gas": r"\bgas\b|\bnwy\b",
    # removing the \b at the end to get "electric", "electricity" et "electricaire"
    "heat_energy_electric": r"\belectric", 
    "heat_energy_oil": r"\boil\b",
    "heat_energy_solid": r"\bsolid\b|\bwood\b|\bcoal\b",
    
    # CLASSIC SYSTEM
    "heat_sys_boiler": r"\bboiler\b|\bbwyler\b", # Ajout du gallois
    "heat_sys_radiator": r"\bradiators?\b|\brheiddiaduron\b", # Ajout du gallois
    "heat_sys_room_heater": r"\broom heaters?\b",
    "heat_sys_storage": r"\bstorage\b",
    "heat_sys_underfloor": r"\bunderfloor\b",
    "heat_sys_heat_pump": r"\bheat pumps?\b", # Ajout du s optionnel (?)
    "heat_sys_community": r"\bcommunity\b",
    
    # NEW SYSTEMS DETECTED
    "heat_sys_warm_air": r"\bwarm air\b",
    "heat_sys_portable_or_none": r"\bno system\b|\bportable\b",
    "heat_sys_ceiling": r"\bceiling\b",
    "heat_sys_micro_cogen": r"micro-cogeneration",
    "heat_sys_hot_water_only": r"hot-water-only"
}

# Creating columns with the new systems detected
for col_name, pattern in keywords_mapping_v2.items():
    df_model[col_name] = heat_desc.str.contains(pattern, regex=True).astype(int)

In [7]:
colonnes_systemes = [col for col in keywords_mapping_v2.keys() if col.startswith("heat_sys_")]

# Sum by row, if sum = 0, no system detected
lignes_sans_systeme = df_model[df_model[colonnes_systemes].sum(axis=1) == 0]

# printing descriptions to see what we missed
print("Descriptions non capturées :")
print(lignes_sans_systeme["MAINHEAT_DESCRIPTION"].value_counts())

Descriptions non capturées :
MAINHEAT_DESCRIPTION
SAP05:Main-Heating    545
Name: count, dtype: int64


In [8]:
# 2. Listing every columns about SYSTEMS (Not ENERGY)
colonnes_systemes = [col for col in df_model.columns if col.startswith("heat_sys_")]

# 3. Creating column "Unknown" : 
# = True (1) if the sum of every system is 0, else False (0)
df_model["heat_sys_unknown"] = (df_model[colonnes_systemes].sum(axis=1) == 0).astype(int)

# --- Final check ---
print("System repartition (1 = present) :")
toutes_colonnes_sys = colonnes_systemes + ["heat_sys_unknown"]
print(df_model[toutes_colonnes_sys].sum().sort_values(ascending=False))

System repartition (1 = present) :
heat_sys_radiator            232575
heat_sys_boiler              231123
heat_sys_room_heater          57496
heat_sys_storage              22204
heat_sys_community            18067
heat_sys_heat_pump             2167
heat_sys_underfloor            1964
heat_sys_portable_or_none      1575
heat_sys_unknown                545
heat_sys_warm_air               323
heat_sys_ceiling                 10
heat_sys_micro_cogen              5
heat_sys_hot_water_only           1
dtype: int64


In [9]:
heat_desc = df_model["MAINHEAT_DESCRIPTION"].str.lower().fillna("")

# 2. Creating binary columns for ENERGY
df_model["heat_energy_gas"] = heat_desc.str.contains("gas").astype(int)
df_model["heat_energy_electric"] = heat_desc.str.contains("electric").astype(int)

# 3. Creating binary columns for systems
df_model["heat_sys_boiler"] = heat_desc.str.contains("boiler").astype(int)
df_model["heat_sys_room_heaters"] = heat_desc.str.contains("room heaters").astype(int)
df_model["heat_sys_storage"] = heat_desc.str.contains("storage").astype(int)
df_model["heat_sys_heat_pump"] = heat_desc.str.contains("heat pump").astype(int)
df_model["heat_sys_underfloor"] = heat_desc.str.contains("underfloor").astype(int)
df_model["heat_sys_community"] = heat_desc.str.contains("community scheme").astype(int)

created_col = [col for col in df_model.columns if col.startswith("heat_")]
print(df_model[["MAINHEAT_DESCRIPTION"] + created_col].head())

              MAINHEAT_DESCRIPTION  heat_energy_gas  heat_energy_electric  \
0  Boiler and radiators, mains gas                1                     0   
1  Boiler and radiators, mains gas                1                     0   
2  Boiler and radiators, mains gas                1                     0   
3           Room heaters, electric                0                     1   
4  Boiler and radiators, mains gas                1                     0   

   heat_energy_oil  heat_energy_solid  heat_sys_boiler  heat_sys_radiator  \
0                0                  0                1                  1   
1                0                  0                1                  1   
2                0                  0                1                  1   
3                0                  0                0                  0   
4                0                  0                1                  1   

   heat_sys_room_heater  heat_sys_storage  heat_sys_underfloor  \
0       

#### ROOF_DESCRIPTION

In [10]:
df_model['ROOF_DESCRIPTION'].value_counts()

ROOF_DESCRIPTION
(another dwelling above)                     82506
Pitched, 200 mm loft insulation              31932
Pitched, no insulation (assumed)             30498
(other premises above)                       26512
Pitched, 250 mm loft insulation              25728
                                             ...  
Ar oleddf, dim inswleiddio                       1
Average thermal transmittance 0.46 W/m-¦K        1
Average thermal transmittance 0.67 W/m-¦K        1
Roof room(s), limited insulation                 1
Average thermal transmittance 0.03 W/m-¦K        1
Name: count, Length: 223, dtype: int64

In [11]:
# 1. PREPROCESSING: Clean the text column
# ---------------------------------------------------------
roof_desc = df_model['ROOF_DESCRIPTION'].str.lower().fillna('')

# ---------------------------------------------------------
# 2. FEATURE EXTRACTION: Binary categorical columns (1/0)
# ---------------------------------------------------------
# Extract roof topology (Added 'thatched' roofs)
df_model['roof_type_pitched'] = roof_desc.str.contains(r'\bpitched\b|\bar oleddf\b', regex=True).astype(int)
df_model['roof_type_room'] = roof_desc.str.contains(r'\broof room\b', regex=True).astype(int)
df_model['roof_type_flat'] = roof_desc.str.contains(r'\bflat\b', regex=True).astype(int)
df_model['roof_type_thatched'] = roof_desc.str.contains(r'\bthatched\b', regex=True).astype(int)

# Extract "Covered Above" 
df_model['roof_covered_above'] = roof_desc.str.contains(r'dwelling above|premises above', regex=True).astype(int)

# Extract specific insulation status (Added 'additional insulation')
df_model['roof_insulation_none'] = roof_desc.str.contains(r'no insulation|dim inswleiddio', regex=True).astype(int)
df_model['roof_insulation_limited'] = roof_desc.str.contains(r'limited insulation', regex=True).astype(int)
df_model['roof_insulated_generic'] = roof_desc.str.contains(r'\binsulated\b|additional insulation', regex=True).astype(int)

# Extract SAP05 indicator (Missing data handled by software default)
df_model['roof_unknown'] = roof_desc.str.contains(r'sap05', regex=True).astype(int)

# ---------------------------------------------------------
# 3. FEATURE EXTRACTION: Continuous numerical column
# ---------------------------------------------------------
df_model['roof_insulation_thickness_mm'] = roof_desc.str.extract(r'(\d+)\s*mm').astype(float)

# ---------------------------------------------------------
# 4. DATA IMPUTATION: Handle missing numerical values (NaN)
# ---------------------------------------------------------
# Rule A: "no insulation" -> 0 mm
df_model['roof_insulation_thickness_mm'] = np.where(
    df_model['roof_insulation_none'] == 1,
    0.0,
    df_model['roof_insulation_thickness_mm']
)

# Rule B: "limited insulation" -> 50 mm
df_model['roof_insulation_thickness_mm'] = np.where(
    df_model['roof_insulation_limited'] == 1,
    50.0,
    df_model['roof_insulation_thickness_mm']
)

# Rule C: Covered above -> 300 mm proxy
df_model['roof_insulation_thickness_mm'] = np.where(
    df_model['roof_covered_above'] == 1,
    300.0,
    df_model['roof_insulation_thickness_mm']
)

# Rule D: Fallback for NaNs 
median_thickness = df_model['roof_insulation_thickness_mm'].median()
df_model['roof_insulation_thickness_mm'] = df_model['roof_insulation_thickness_mm'].fillna(median_thickness)

# ---------------------------------------------------------
# 5. VERIFICATION: Did we miss any "black swans"?
# ---------------------------------------------------------
# Updated binary features list with the new columns
binary_features = [
    'roof_type_pitched', 
    'roof_type_room', 
    'roof_type_flat',
    'roof_type_thatched',
    'roof_covered_above', 
    'roof_insulation_none', 
    'roof_insulation_limited',
    'roof_insulated_generic',
    'roof_unknown'
]

# Check condition
is_handled = (
    (df_model[binary_features].sum(axis=1) > 0) | 
    roof_desc.str.contains('mm|transmittance', na=False)
)

unhandled_rows = df_model[~is_handled]

print("--- VERIFICATION RESULTS ---")
print(f"Total rows processed: {len(df_model)}")
print(f"Rows completely unhandled by rules: {len(unhandled_rows)}")

if len(unhandled_rows) > 0:
    print("\nTop unhandled descriptions (Inspect these to see if a rule is missing):")
    print(unhandled_rows['ROOF_DESCRIPTION'].value_counts().head(10))
else:
    print("\nSuccess! 100% of the rows have been captured by our rules or intentionally ignored.")

--- VERIFICATION RESULTS ---
Total rows processed: 334270
Rows completely unhandled by rules: 41

Top unhandled descriptions (Inspect these to see if a rule is missing):
Series([], Name: count, dtype: int64)


#### WALLS_DESCRIPTION

In [12]:
df_model['WALLS_DESCRIPTION'].value_counts()

WALLS_DESCRIPTION
Cavity wall, filled cavity                               92385
Cavity wall, as built, no insulation (assumed)           56011
Solid brick, as built, no insulation (assumed)           51521
Cavity wall, as built, insulated (assumed)               41241
System built, as built, insulated (assumed)              20827
                                                         ...  
Average thermal transmittance 1.85 W/m-¦K                    1
Average thermal transmittance 0.07 W/m-¦K                    1
Average thermal transmittance 0.85 W/m-¦K                    1
Curtain Wall, as built, no insulation (assumed)              1
Basement wall, as built, partial insulation (assumed)        1
Name: count, Length: 399, dtype: int64

In [13]:
# 1. Count how many rows contain the word "transmittance" in WALLS_DESCRIPTION
walls_u_value_mask = df_model['WALLS_DESCRIPTION'].str.lower().str.contains('transmittance', na=False)
walls_u_value_count = walls_u_value_mask.sum()

# 2. Calculate the percentage relative to the whole dataset
total_rows = len(df_model)
walls_u_value_percentage = (walls_u_value_count / total_rows) * 100

# 3. Display the results
print(f"Total rows in dataset: {total_rows}")
print(f"Walls with a precise U-value: {walls_u_value_count}")
print(f"Percentage: {walls_u_value_percentage:.3f}%")

Total rows in dataset: 334270
Walls with a precise U-value: 43170
Percentage: 12.915%


In [14]:
# ---------------------------------------------------------
# 1. PREPROCESSING: Clean the text column
# ---------------------------------------------------------
walls_desc = df_model['WALLS_DESCRIPTION'].str.lower().fillna('')

# ---------------------------------------------------------
# 2. FEATURE EXTRACTION: Binary categorical columns (1/0)
# ---------------------------------------------------------
# --- Wall Topology & Materials ---
df_model['wall_type_cavity'] = walls_desc.str.contains(r'\bcavity\b', regex=True).astype(int)
df_model['wall_type_solid'] = walls_desc.str.contains(r'\bsolid\b', regex=True).astype(int)
df_model['wall_type_system'] = walls_desc.str.contains(r'\bsystem built\b', regex=True).astype(int)
df_model['wall_type_timber'] = walls_desc.str.contains(r'\btimber\b', regex=True).astype(int)
df_model['wall_type_curtain'] = walls_desc.str.contains(r'\bcurtain\b', regex=True).astype(int)
df_model['wall_type_basement'] = walls_desc.str.contains(r'\bbasement\b', regex=True).astype(int)
# New additions for traditional/specific builds
df_model['wall_type_stone'] = walls_desc.str.contains(r'\b(sandstone|limestone|granite|whin|whinstone)\b', regex=True).astype(int)
df_model['wall_type_cob'] = walls_desc.str.contains(r'\bcob\b', regex=True).astype(int)
df_model['wall_type_park_home'] = walls_desc.str.contains(r'\bpark home\b', regex=True).astype(int)

# --- Insulation Status ---
df_model['wall_insulation_none'] = walls_desc.str.contains(r'no insulation|dim inswleiddio', regex=True).astype(int)
df_model['wall_insulation_filled_cavity'] = walls_desc.str.contains(r'filled cavity', regex=True).astype(int)
df_model['wall_insulation_partial'] = walls_desc.str.contains(r'partial insulation', regex=True).astype(int)
# New specific insulation locations
df_model['wall_insulation_internal'] = walls_desc.str.contains(r'internal insulation', regex=True).astype(int)
df_model['wall_insulation_external'] = walls_desc.str.contains(r'external insulation', regex=True).astype(int)
# Fallback generic insulation
df_model['wall_insulated_generic'] = walls_desc.str.contains(r'\binsulated\b', regex=True).astype(int)

# --- Missing / Default Indicators ---
df_model['wall_unknown'] = walls_desc.str.contains(r'sap05', regex=True).astype(int)
df_model['wall_measured_u_value'] = walls_desc.str.contains(r'transmittance', regex=True).astype(int)

# ---------------------------------------------------------
# 3. VERIFICATION
# ---------------------------------------------------------
binary_features_walls = [
    'wall_type_cavity', 'wall_type_solid', 'wall_type_system', 
    'wall_type_timber', 'wall_type_curtain', 'wall_type_basement',
    'wall_type_stone', 'wall_type_cob', 'wall_type_park_home',
    'wall_insulation_none', 'wall_insulation_filled_cavity', 
    'wall_insulation_partial', 'wall_insulation_internal',
    'wall_insulation_external', 'wall_insulated_generic', 
    'wall_unknown', 'wall_measured_u_value'
]

# Check if handled
is_handled_walls = (
    (df_model[binary_features_walls].sum(axis=1) > 0) | 
    (walls_desc == '')
)

unhandled_rows_walls = df_model[~is_handled_walls]

print("--- VERIFICATION RESULTS FOR WALLS ---")
print(f"Total rows processed: {len(df_model)}")
print(f"Rows completely unhandled by rules: {len(unhandled_rows_walls)}")

if len(unhandled_rows_walls) == 0:
    print("\n100% of the walls are now categorized perfectly.")

/tmp/ipykernel_105257/3285685736.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_model['wall_type_stone'] = walls_desc.str.contains(r'\b(sandstone|limestone|granite|whin|whinstone)\b', regex=True).astype(int)


--- VERIFICATION RESULTS FOR WALLS ---
Total rows processed: 334270
Rows completely unhandled by rules: 0

100% of the walls are now categorized perfectly.


#### WINDOWS_DESCRIPTION

In [15]:
df_model['WINDOWS_DESCRIPTION'].value_counts()

WINDOWS_DESCRIPTION
Fully double glazed                                       255361
High performance glazing                                   41130
Single glazed                                              15155
Partial double glazing                                      9907
Mostly double glazing                                       6838
Some double glazing                                         2689
Full secondary glazing                                      1422
Fully triple glazed                                          419
SAP05:Windows                                                306
Multiple glazing throughout                                  202
Partial secondary glazing                                    195
Single glazeddouble glazing                                  156
Fully secondary glazing                                       78
Mostly multiple glazing                                       57
Partial multiple glazing                                      48
Some 

In [16]:
# ---------------------------------------------------------
# 1. PREPROCESSING: Clean the text column
# ---------------------------------------------------------
windows_desc = df_model['WINDOWS_DESCRIPTION'].str.lower().fillna('')

# ---------------------------------------------------------
# 2. FEATURE EXTRACTION: Binary categorical columns (1/0)
# ---------------------------------------------------------
# --- A. Glazing Technologies ---
# We avoid '\b' on English words to catch typos like 'glazeddouble'
df_model['win_glazing_single'] = windows_desc.str.contains(r'single', regex=True).astype(int)
df_model['win_glazing_double'] = windows_desc.str.contains(r'double|dwbl', regex=True).astype(int)
df_model['win_glazing_triple'] = windows_desc.str.contains(r'triple', regex=True).astype(int)
df_model['win_glazing_secondary'] = windows_desc.str.contains(r'secondary', regex=True).astype(int)
df_model['win_glazing_multiple'] = windows_desc.str.contains(r'multiple|lluosog', regex=True).astype(int)
df_model['win_glazing_high_perf'] = windows_desc.str.contains(r'high performance|perfformiad uchel', regex=True).astype(int)

# --- B. Coverage Extent (Proxy for continuous impact) ---
df_model['win_extent_full'] = windows_desc.str.contains(r'fully?|throughout|llawn|ym mhobman', regex=True).astype(int)
df_model['win_extent_partial'] = windows_desc.str.contains(r'partial|mostly|some', regex=True).astype(int)

# --- C. Missing / Default Indicators ---
df_model['win_unknown'] = windows_desc.str.contains(r'sap05', regex=True).astype(int)

# ---------------------------------------------------------
# 3. VERIFICATION
# ---------------------------------------------------------
binary_features_windows = [
    'win_glazing_single', 'win_glazing_double', 'win_glazing_triple',
    'win_glazing_secondary', 'win_glazing_multiple', 'win_glazing_high_perf',
    'win_unknown'
]

# We consider it handled if at least one glazing tech is detected OR if it's empty
is_handled_windows = (
    (df_model[binary_features_windows].sum(axis=1) > 0) | 
    (windows_desc == '')
)

unhandled_rows_windows = df_model[~is_handled_windows]

print("--- VERIFICATION RESULTS FOR WINDOWS ---")
print(f"Total rows processed: {len(df_model)}")
print(f"Rows completely unhandled by rules: {len(unhandled_rows_windows)}")

if len(unhandled_rows_windows) > 0:
    print("\nTop unhandled descriptions:")
    # Using dropna=False to show if there are hidden NaNs
    print(unhandled_rows_windows['WINDOWS_DESCRIPTION'].value_counts(dropna=False).head(15))
else:
    print("\n100% of the windows are now categorized perfectly.")

--- VERIFICATION RESULTS FOR WINDOWS ---
Total rows processed: 334270
Rows completely unhandled by rules: 0

100% of the windows are now categorized perfectly.


#### PROPERTY_TYPE

In [17]:
df_model['PROPERTY_TYPE'].value_counts()

PROPERTY_TYPE
House         178186
Flat          147250
Maisonette      4747
Bungalow        4086
Park home          1
Name: count, dtype: int64

In [18]:
# 1. Clean the text column
prop_type = df_model['PROPERTY_TYPE'].str.lower().fillna('')

# 2. Extract standard property types (One-Hot Encoding style)
df_model['prop_type_house'] = prop_type.str.contains(r'\bhouse\b', regex=True).astype(int)
df_model['prop_type_flat'] = prop_type.str.contains(r'\bflat\b', regex=True).astype(int)
df_model['prop_type_maisonette'] = prop_type.str.contains(r'\bmaisonette\b', regex=True).astype(int)

# 3. Group Bungalow and Park Home together (due to Park Home being a single outlier)
df_model['prop_type_bungalow_or_park'] = prop_type.str.contains(r'\bbungalow\b|\bpark home\b', regex=True).astype(int)

# --- VERIFICATION ---
binary_features_prop = [
    'prop_type_house', 'prop_type_flat', 
    'prop_type_maisonette', 'prop_type_bungalow_or_park'
]

# Check if handled
is_handled_prop = (df_model[binary_features_prop].sum(axis=1) > 0) | (prop_type == '')
unhandled_rows_prop = df_model[~is_handled_prop]

print("--- VERIFICATION RESULTS FOR PROPERTY TYPE ---")
print(f"Rows completely unhandled: {len(unhandled_rows_prop)}")

--- VERIFICATION RESULTS FOR PROPERTY TYPE ---
Rows completely unhandled: 0


#### BUILT_FORM

In [19]:
df_model['BUILT_FORM'].value_counts()

BUILT_FORM
Mid-Terrace             120996
Semi-Detached            93173
End-Terrace              60068
Detached                 31115
NO DATA!                  9367
Enclosed End-Terrace      8806
Enclosed Mid-Terrace      6157
Not Recorded              4224
Name: count, dtype: int64

In [20]:
# ---------------------------------------------------------
# 1. PREPROCESSING: Clean the text column
# ---------------------------------------------------------
built_form = df_model['BUILT_FORM'].str.lower().fillna('')

# ---------------------------------------------------------
# 2. FEATURE EXTRACTION: Base topologies & Modifiers
# ---------------------------------------------------------
# --- Base Topology ---
# We use exact word catching to avoid 'detached' triggering on 'semi-detached'
df_model['form_semi_detached'] = built_form.str.contains(r'semi-detached', regex=True).astype(int)

# Detached: Contains 'detached' but IS NOT semi-detached
df_model['form_detached'] = (
    built_form.str.contains(r'detached', regex=True) & 
    (df_model['form_semi_detached'] == 0)
).astype(int)

# Terraces (Mid and End) - This will catch the 'enclosed' ones too!
df_model['form_mid_terrace'] = built_form.str.contains(r'mid-terrace', regex=True).astype(int)
df_model['form_end_terrace'] = built_form.str.contains(r'end-terrace', regex=True).astype(int)

# --- Modifiers ---
df_model['form_enclosed'] = built_form.str.contains(r'enclosed', regex=True).astype(int)

# --- Missing Data ---
df_model['form_unknown'] = built_form.str.contains(r'no data|not recorded', regex=True).astype(int)

# ---------------------------------------------------------
# 3. VERIFICATION
# ---------------------------------------------------------
binary_features_form = [
    'form_detached', 'form_semi_detached', 
    'form_mid_terrace', 'form_end_terrace', 
    'form_unknown'
]

# A row is handled if it has a base topology or is explicitly unknown
is_handled_form = (df_model[binary_features_form].sum(axis=1) > 0) | (built_form == '')
unhandled_rows_form = df_model[~is_handled_form]

print("--- VERIFICATION RESULTS FOR BUILT FORM ---")
print(f"Total rows processed: {len(df_model)}")
print(f"Rows completely unhandled by rules: {len(unhandled_rows_form)}")

if len(unhandled_rows_form) > 0:
    print("\nTop unhandled descriptions:")
    print(unhandled_rows_form['BUILT_FORM'].value_counts(dropna=False).head(10))
else:
    print("\n100% of the built forms are cleanly categorized.")

--- VERIFICATION RESULTS FOR BUILT FORM ---
Total rows processed: 334270
Rows completely unhandled by rules: 0

100% of the built forms are cleanly categorized.


#### CONSTRUCTION_AGE_BAND

In [21]:
df_model['CONSTRUCTION_AGE_BAND'].value_counts()

CONSTRUCTION_AGE_BAND
England and Wales: 1900-1929       66641
England and Wales: 1930-1949       52276
England and Wales: 1950-1966       34866
England and Wales: 2003-2006       24620
NO DATA!                           23464
England and Wales: 1967-1975       22428
England and Wales: before 1900     19982
England and Wales: 1996-2002       17884
England and Wales: 1976-1982       14997
England and Wales: 1983-1990       10766
England and Wales: 1991-1995       10028
England and Wales: 2007 onwards     7458
2021                                4842
2023                                3875
England and Wales: 2007-2011        3821
2022                                2458
2025                                2304
England and Wales: 2012 onwards     2248
2020                                1324
2024                                1040
England and Wales: 2012-2021         626
England and Wales: 2022 onwards      589
2018                                 499
INVALID!                           

In [22]:
# ---------------------------------------------------------
# 1. PREPROCESSING
# ---------------------------------------------------------
age_band = df_model['CONSTRUCTION_AGE_BAND'].str.lower().fillna('')

# ---------------------------------------------------------
# 2. FEATURE EXTRACTION: Missing data flags
# ---------------------------------------------------------
df_model['construction_age_unknown'] = age_band.str.contains(r'no data|invalid', regex=True).astype(int)

# ---------------------------------------------------------
# 3. FEATURE EXTRACTION: Continuous Estimated Year (Midpoint)
# ---------------------------------------------------------
# Extract up to two 4-digit years from the text
# Group 1 captures the first year, Group 2 captures the second year (if it exists)
extracted_years = age_band.str.extract(r'(\d{4})(?:.*?(\d{4}))?')
extracted_years.columns = ['start_year', 'end_year']

# Convert to float for mathematical operations
start_year = extracted_years['start_year'].astype(float)
end_year = extracted_years['end_year'].astype(float)

# Calculate the midpoint: if end_year exists, average them. Otherwise, just use start_year.
df_model['construction_year_estimated'] = np.where(
    end_year.notna(),
    (start_year + end_year) / 2.0,
    start_year
)

# Correct the "before 1900" edge case
is_before_1900 = age_band.str.contains('before 1900', na=False)
df_model['construction_year_estimated'] = np.where(
    is_before_1900, 
    1850.0, 
    df_model['construction_year_estimated']
)

# ---------------------------------------------------------
# 4. DATA IMPUTATION
# ---------------------------------------------------------
# Impute the missing values (NO DATA!, INVALID!) with the median of our new calculated column
median_year = df_model['construction_year_estimated'].median()
df_model['construction_year_estimated'] = df_model['construction_year_estimated'].fillna(median_year)

# ---------------------------------------------------------
# 5. VERIFICATION
# ---------------------------------------------------------
print("--- VERIFICATION RESULTS FOR CONSTRUCTION AGE ---")
print(f"Total rows processed: {len(df_model)}")

# Verify the transformation specifically on the tricky bands
verify_cols = ['CONSTRUCTION_AGE_BAND', 'construction_year_estimated']
print("\nTransformation sample (Check the midpoints!):")
sample_mask = age_band.str.contains('1930-1949|2007 onwards|before 1900|2021', na=False)
sample_to_check = df_model.loc[sample_mask, verify_cols].drop_duplicates(subset=['CONSTRUCTION_AGE_BAND'])
print(sample_to_check)

--- VERIFICATION RESULTS FOR CONSTRUCTION AGE ---
Total rows processed: 334270

Transformation sample (Check the midpoints!):
                  CONSTRUCTION_AGE_BAND  construction_year_estimated
0          England and Wales: 1930-1949                       1939.5
19                                 2021                       2021.0
54      England and Wales: 2007 onwards                       2007.0
67       England and Wales: before 1900                       1850.0
318749     England and Wales: 2012-2021                       2016.5


In [23]:
df_clean = df_model.copy()

# ---------------------------------------------------------
# 1. DROP ORIGINAL TEXT COLUMNS
# ---------------------------------------------------------
# List of the raw text columns we have successfully engineered
cols_to_drop = [
    'ROOF_DESCRIPTION',
    'WALLS_DESCRIPTION',
    'WINDOWS_DESCRIPTION',
    'PROPERTY_TYPE',
    'BUILT_FORM',
    'CONSTRUCTION_AGE_BAND',
    'MAINHEAT_DESCRIPTION'
    # Feel free to add 'HEATING_SYSTEM' or any other processed column here
]

# Drop these columns ONLY from our new clean dataframe
df_clean = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

# ---------------------------------------------------------
# 2. DATASET HEALTH CHECK
# ---------------------------------------------------------
print("=== DATASET HEALTH CHECK BEFORE MACHINE LEARNING ===\n")

# A. Final dimensions check
print(f"-> Final dimensions: {df_clean.shape[0]} rows and {df_clean.shape[1]} columns (features).\n")

# B. Missing values (NaNs) check
missing_values = df_clean.isna().sum()
missing_cols = missing_values[missing_values > 0]

print("-> 1. Missing Values (NaNs) Check:")
if missing_cols.empty:
    print("   [SUCCESS] Perfect! There are no missing values left in the dataset.")
else:
    print("   [WARNING] There are still NaNs in the following columns:")
    print(missing_cols)

# C. Data types (dtypes) check
# Looking for any columns that are NOT numeric (int or float)
non_numeric_cols = df_clean.select_dtypes(exclude=[np.number]).columns

print("\n-> 2. Data Formats (Numeric) Check:")
if len(non_numeric_cols) == 0:
    print("   [SUCCESS] 100% of the columns are numeric (int/float). The dataset is ready for modeling!")
else:
    print("   [WARNING] These columns are still text/categorical and will cause ML algorithms to fail:")
    for col in non_numeric_cols:
        print(f"   - {col} (Type: {df_clean[col].dtype})")

=== DATASET HEALTH CHECK BEFORE MACHINE LEARNING ===

-> Final dimensions: 334270 rows and 71 columns (features).

-> 1. Missing Values (NaNs) Check:
   [SUCCESS] Perfect! There are no missing values left in the dataset.

-> 2. Data Formats (Numeric) Check:
   [WARNING] These columns are still text/categorical and will cause ML algorithms to fail:
   - CURRENT_ENERGY_RATING (Type: object)


In [ ]:
df_clean

,TOTAL_FLOOR_AREA,CURRENT_ENERGY_RATING,CURRENT_ENERGY_EFFICIENCY,ENERGY_CONSUMPTION_CURRENT,target_is_efficient,heat_energy_gas,heat_energy_electric,heat_energy_oil,heat_energy_solid,heat_sys_boiler,...,prop_type_maisonette,prop_type_bungalow_or_park,form_semi_detached,form_detached,form_mid_terrace,form_end_terrace,form_enclosed,form_unknown,construction_age_unknown,construction_year_estimated
0,81.00,C,70,209,1,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,1939.5
1,88.92,D,64,279,0,1,0,0,0,1,...,0,0,1,0,0,0,0,0,0,1939.5
2,59.00,D,55,322,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,1914.5
3,156.00,G,11,505,0,0,1,0,0,0,...,0,0,0,0,0,1,1,0,0,1971.0
4,120.60,C,74,182,1,1,0,0,0,1,...,0,0,0,0,0,1,0,0,0,1971.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334265,76.00,C,78,120,1,1,0,0,0,1,...,0,0,0,0,0,0,0,1,0,1914.5
334266,81.00,C,71,154,1,1,0,0,0,1,...,0,0,0,0,0,1,0,0,0,1979.0
334267,75.00,D,66,225,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,1914.5
334268,65.00,C,74,143,1,1,0,0,0,1,...,0,0,0,0,0,0,0,1,0,1939.5
